In [ ]:
import json
from time import sleep

import GPUtil
import numpy as np
import pandas as pd
import psutil
import torch
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader

import internal.utils.telegram_utils as tg
from internal.data_types import PainDataset
from internal.nn.models.pain_tcn_bilstm_attn import PainTCNBiLSTMAttn
from internal.nn.training.pain_tcn_bilstm_attn_trainer import PainTCNBiLSTMAttnTrainer
from internal.persistence_manager import PersistenceManager

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
@torch.no_grad()
def infer_logits(_model, _x_num, _x_surv, _x_sta, _x_summ, batch_size=64):
    _model.eval()
    ds = PainDataset(_x_num, _x_surv, _x_sta, _x_summ)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    predictions = []
    for b in dl:
        x_num  = b['x_num'].to(device)
        x_sta  = b['x_sta'].to(device)
        x_surv = [s.to(device) for s in b['x_surv']]
        x_summ = b['x_summ'].to(device)
        logits = _model(x_num, x_surv, x_sta, x_summ)  # (B,3)
        predictions.append(logits.detach().cpu().numpy())
    return np.concatenate(predictions, axis=0)  # (N,3)

@torch.no_grad()
def infer_logits_tta(_model, _x_num, _x_surv, _x_sta, _x_summ, _n_aug=5, _noise_std=0.01, _roll=2) -> np.ndarray:
    _model.eval()
    outs = []
    for _k in range(_n_aug):
        xn = _x_num.copy()
        if _k > 0:
            # small Gaussian jitter on numeric channels
            xn += np.random.normal(0, _noise_std, size=xn.shape).astype(xn.dtype)
            # tiny circular time shift
            shift = np.random.randint(-_roll, _roll + 1)
            if shift != 0:
                xn = np.roll(xn, shift=shift, axis=1)  # roll along the time
        outs.append(infer_logits(_model, xn, _x_surv, _x_sta, _x_summ, batch_size=64))
    return np.mean(outs, axis=0)  # (N,3)

In [ ]:
from torch.utils.data import WeightedRandomSampler

data = PersistenceManager.load_arrays_v2()

train_dataset = PainDataset(
    data.X_dyn_num_train,
    data.X_surv_train,
    data.X_sta_train,
    data.X_dyn_summ_train,
    data.y_train
)
val_dataset   = PainDataset(
    data.X_dyn_num_val,
    data.X_surv_val,
    data.X_sta_val,
    data.X_dyn_summ_val,
    data.y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,          # randomize samples each epoch
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_fold_logits = []  # to store per-fold test logits for ensembling
fold_scores = []
temperatures = []
# set seed 2021 for reproducibility
seed = 2021
np.random.seed(seed)
torch.manual_seed(seed)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
y_all = np.concatenate([data.y_train, data.y_val], axis=0)
X_num_all = np.concatenate([data.X_dyn_num_train, data.X_dyn_num_val], axis=0) # (N,160,90)
X_sta_all = np.concatenate([data.X_sta_train, data.X_sta_val], axis=0)       # (N,7)
X_surv_all = [np.concatenate([data.X_surv_train[i], data.X_surv_val[i]], axis=0) for i in range(4)]
X_summ_all = np.concatenate([data.X_dyn_summ_train, data.X_dyn_summ_val], axis=0) # (N,F_summ)
class2_multipliers = [1, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3]
results: dict[str, dict[str, float]] = {}
best_f1 = 0.0

trainer = PainTCNBiLSTMAttnTrainer(
    classes=[0, 1, 2],
    epochs=1000,
    patience=28,
    warmup_epochs=20,
    p_drop_summ=0.3,
    summary_window=data.feat_eng.window,
    device=device
)

tg.send_message("🚀 Starting K-Fold training for PainTCNBiLSTMAttn model...", False)
for class2_multiplier in class2_multipliers:
    for k, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
        print(f"=== Fold {k} with class-2 multiplier {class2_multiplier} ===")
        Xtr_num, Xva_num = X_num_all[tr_idx], X_num_all[va_idx]
        Xtr_sta, Xva_sta = X_sta_all[tr_idx], X_sta_all[va_idx]
        Xtr_surv = [s[tr_idx] for s in X_surv_all]
        Xva_surv = [s[va_idx] for s in X_surv_all]
        Xtr_summ, Xva_summ = X_summ_all[tr_idx], X_summ_all[va_idx]
        ytr, yva = y_all[tr_idx], y_all[va_idx]

        train_ds = PainDataset(Xtr_num, Xtr_surv, Xtr_sta, Xtr_summ, ytr)
        val_ds   = PainDataset(Xva_num, Xva_surv, Xva_sta, Xva_summ, yva)

        class_counts = np.bincount(ytr, minlength=3)
        w_class = 1.0 / np.clip(class_counts, 1, None)
        sample_w = w_class[ytr]

        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_w, dtype=torch.double), # CPU
            num_samples=len(ytr),
            replacement=True
        )

        train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, sampler=sampler,  num_workers=4, pin_memory=True)
        # do not shuffle validation since we want consistent metrics across epochs
        val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

        model, f1, best_b, t_fold = trainer.train_one_fold(
            model=PainTCNBiLSTMAttn(
                d_num=90,
                d_emb=2,
                d_sta=7,
                d_summ=data.X_dyn_summ_train.shape[1],
                tcn_channels=128,
                lstm_hidden=128,
                num_classes=3,
                dropout=0.3
            ).to(device),
            k=k, train_loader=train_loader, val_loader=val_loader, y_train=ytr, y_val=yva,
            class_multipliers={2: class2_multiplier}
        )
        fold_scores.append(f1)
        temperatures.append(t_fold)

        # ---- (optional) test logits for ensembling ----
        logits_te = infer_logits_tta(
            model,
            data.X_dyn_num_test, data.X_surv_test, data.X_sta_test, data.X_dyn_summ_test,
            _n_aug=5, _noise_std=0.01, _roll=2
        ).astype(np.float32)  # (N_test, 3)
        # 1) temperature scaling (per-fold scalar)
        logits_te = logits_te / float(t_fold)
        # 2) per-class bias (shape (3,))
        logits_te = logits_te + best_b[np.newaxis, :]  # or best_b[None, :]
        # store for ensembling
        test_fold_logits.append(logits_te)

        # if the f1 is the greatest so far, save the model as the "best overall"
        if f1 > best_f1:
            best_f1 = f1
            filename = f'artifacts/best_overall_PainTCNBiLSTMAttn_K{k}-mul_{class2_multiplier}-f1_{best_f1:.6f}-seed_{seed}'
            torch.save(model.state_dict(), f'{filename}.pt')
            try:
                json.dump({
                    'macroF':f1,
                    'window': data.feat_eng['window']
                }, open(f'{filename}.json', 'w'), indent=4)
            except Exception as e:
                print("Could not save best overall metrics json:", e)
            tg.send_file(
                f'{filename}.pt',
                caption=f"🏆 New best overall PainTCNBiLSTMAttn model: Fold {k} class2_mul {class2_multiplier} macro-F1 {best_f1}",
                raise_on_failure=False
            )

        # record fold results
        results[f'fold_{k}_class2mul_{class2_multiplier}'] = {
            'macroF1': f1,
            'best_bias': best_b.tolist(),
            'temperature': t_fold
        }

        # notify me about fold completion with a markdown summary
        tg.send_message(
            f"✅ Completed Fold {k} with class-2 multiplier {class2_multiplier}\n"
            f"    - Macro-F1: **{f1:.4f}**\n"
            f"    - Best bias: `{best_b.tolist()}`\n"
            f"    - Temperature: **{t_fold:.4f}**",
            raise_on_failure=False
        )

        # --- GPU / CPU stats after each fold ---
        try:
            for gpu in GPUtil.getGPUs():
                print(f"{gpu.name} ({gpu.id}) Usage after fold {k} class2_mul {class2_multiplier}:")
                print(f"    Temp: {gpu.temperature}C, Load: {gpu.load*100:.1f}%, Mem Load: {gpu.memoryUtil*100:.1f}%")
        except Exception as e:
            print("Could not get GPU stats:", e)
        try:
            cores = psutil.sensors_temperatures().get('coretemp', [])
            if cores:
                for core in cores:
                    print(f"CPU Core {core.label} Temp: {core.current}C")
        except Exception as e:
            print("Could not get CPU temperature:", e)
        sleep(3) # cooldown between folds
tg.send_message(
    f"✅ Completed K-Fold training for PainTCNBiLSTMAttn model. CV macro-F1 mean: {float(np.mean(fold_scores)):.4f} std: {float(np.std(fold_scores)):.4f}",
    False
)

m_star: str = max(results, key=lambda _k: results[_k]['macroF1'])
print("Best fold:", m_star, "with macro-F1:", results[m_star])
tg.send_message(
    f"🏅 Best fold: {m_star} with macro-F1: {results[m_star]['macroF1']:.4f}",
    False
)

print("CV macro-F1 mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))
print("Temperatures per fold:", temperatures)
# send a telegram chat notification to notify completion
tg.send_message(f"✅ Fold training complete. CV macro-F1 mean: {float(np.mean(fold_scores)):.4f} std: {float(np.std(fold_scores)):.4f}", False)

In [ ]:
# --- ensemble test logits and write submission ---
if len(test_fold_logits) > 0:
    logits_stack = np.stack(test_fold_logits, axis=0)   # (5,1324,3)
    logits_avg   = logits_stack.mean(axis=0)            # (1324,3)
    pred_idx     = logits_avg.argmax(axis=1)            # (1324,)
    idx_to_label = {0:'no_pain', 1:'low_pain', 2:'high_pain'}
    pred_labels  = [idx_to_label[int(i)] for i in pred_idx]

    ids_test = [f'{i:03}' for i in range(1324)]

    sub = pd.DataFrame({'sample_index': ids_test, 'label': pred_labels})
    _filename = f'PainTCNBiLSTMAttn_v3_warmup_rrelu_ema_bias_l1_drop03_mixup_seed2021'
    sub.to_csv(f'submissions/{_filename}.csv', index=False)
    print("Saved:", f'submissions/{_filename}.csv')
    try:
        tg.send_file(
            f'submissions/{_filename}.csv',
            caption="✅ Submission file generated from CV ensemble.",
            raise_on_failure=False
        )
    except Exception as e:
        print("Could not send telegram file:", e)
